In [1]:
!pip freeze | grep scikit-learn

scikit-learn==1.5.0


In [2]:
!python -V

Python 3.9.25


In [2]:
import pickle
import pandas as pd

In [3]:
with open('model.bin', 'rb') as f_in:
    dv, model = pickle.load(f_in)

type(dv), type(model)

(sklearn.feature_extraction._dict_vectorizer.DictVectorizer,
 sklearn.linear_model._base.LinearRegression)

In [4]:
categorical = ['PULocationID', 'DOLocationID']

def read_data(filename):
    df = pd.read_parquet(filename)
    
    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)].copy()

    df[categorical] = df[categorical].fillna(-1).astype('int').astype('str')
    
    return df

In [5]:
df = read_data('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-03.parquet')

In [6]:
dicts = df[categorical].to_dict(orient='records')
X_val = dv.transform(dicts)
y_pred = model.predict(X_val)

type(y_pred)

numpy.ndarray

In [11]:
year, month = 2023, 3

df['ride_id'] = f'{year:04d}/{month:02d}_' + df.index.astype('str')

In [20]:
df_result = pd.DataFrame()
df_result['ride_id'] = df['ride_id']
df_result['predict_duration'] = y_pred

output_file = f'output/yellow/{year}-{month}.parquet'

df_result.to_parquet(
    output_file,
    engine='pyarrow',
    compression=None,
    index=False
)
